In [1]:
%load_ext autoreload
%autoreload 2

from src.dynamical_systems import *
from src.SINDyModel import *
from src.derivatives import *
from src.helpers import *

import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
##### THIS IS A WORK IN PROGRESS! I WOULD LIKE TO MAKE A MORE GENERALIZED VERSION OF THIS CODE #####

# should be able to pass in

In [2]:
def generate_time_span_vs_dt_matrix(rhs, x0, t_span, time_spans, dts, noise_level,
                                      generate_prediction, generate_more_derivatives, threshold=0.05):

    # initialize the data matrix with None entries
    results = [[None for _ in range(len(time_spans))] for _ in range(len(dts))]

    for i, dt in enumerate(dts):
        print(f"dt: {dt}")

        for j, time_span in enumerate(tqdm(time_spans)):
            result = {}

            # generate evaluation time points
            t_eval = np.arange(t_span[0], t_span[1] + dt, dt)
            result["t_eval"] = t_eval

            # generate training data
            x_train = generate_training_data(rhs, x0, t_eval)
            result["x_train"] = x_train
            
            # generate noisy data
            x_train_noisy = add_noise_to_data(x_train, noise_level)
            result["x_train_noisy"] = x_train_noisy

            # generate the derivative of the noisy data using finite difference
            x_dot_finite_difference = dxdt_finite_difference(x_train_noisy, t_eval)
            result["x_dot_finite_difference"] = x_dot_finite_difference

            # generate the true derivative of the data
            # this is only used for comparison and is not used in the SINDy model
            x_dot_true = rhs(0, x_train)
            result["x_dot_true"] = x_dot_true

            # generate different derivative approximations if needed
            if generate_more_derivatives is True:
                x_dot_poly_fit = dxdt_poly_fit(x_train_noisy, t_eval, poly_deg=3, window_diameter=11)
                result['x_dot_poly_fit'] = x_dot_poly_fit
            
            # create and fit a SINDy model
            model = SINDyModel(n_state_vars=len(x0), threshold=threshold, poly_order=5)
            model.fit(x_train_noisy, x_dot_finite_difference)
            result["model"] = model

            # generate the model prediction if needed
            if generate_prediction is True:
                x_pred = generate_model_prediction(model, x0, t_eval)
                result["x_pred"] = x_pred
            
            # store the data entry
            results[i][j] = result
        
    return results

In [3]:
lotka_volterra_rhs = generate_lotka_volterra_rhs(1, 0.1, 1.5, 0.75)

# noise_levels = [0, 0.0001, 0.001, 0.01, 0.05, 0.1, 0.2, 1]
# dts = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1]
noise_levels = np.logspace(-4, 0, 10).tolist()
dts = np.logspace(-2, 0, 11).tolist()

results = generate_noise_level_vs_dt_matrix(
    rhs=lotka_volterra_rhs,
    x0=(10, 5),
    t_span=(0, 50),
    noise_levels=noise_levels,
    dts=dts,
    generate_prediction=True,
    generate_more_derivatives=False,
)

dt: 0.01


 70%|███████   | 7/10 [00:06<00:02,  1.17it/s]c:\Elliot\docs\education\college\2024-2025\MS project\MS-project\src\SINDyModel.py:9: RuntimeWarning: overflow encountered in power
  return lambda x: np.prod(np.power(x, power_tuple), axis=1)
c:\Elliot\docs\education\college\2024-2025\MS project\MS-project\src\SINDyModel.py:141: RuntimeWarning: invalid value encountered in matmul
  x_dot = Theta @ self.Xi
c:\Users\Elliot\anaconda3\envs\ds312\Lib\site-packages\scipy\integrate\_ivp\lsoda.py:161: UserWarning: lsoda: Excess accuracy requested (tolerances too small).
  solver._y, solver.t = integrator.run(
100%|██████████| 10/10 [00:12<00:00,  1.21s/it]


dt: 0.015848931924611134


100%|██████████| 10/10 [00:11<00:00,  1.20s/it]


dt: 0.025118864315095794


...

 20%|██        | 2/10 [00:00<00:02,  2.73it/s]c:\Users\Elliot\anaconda3\envs\ds312\Lib\site-packages\scipy\integrate\_ivp\lsoda.py:161: UserWarning: lsoda: Repeated error test failures (internal error).
  solver._y, solver.t = integrator.run(
100%|██████████| 10/10 [00:13<00:00,  1.38s/it]


dt: 0.6309573444801934


100%|██████████| 10/10 [00:08<00:00,  1.11it/s]


dt: 1.0


 30%|███       | 3/10 [00:13<00:23,  3.42s/it]c:\Users\Elliot\anaconda3\envs\ds312\Lib\site-packages\scipy\integrate\_ivp\lsoda.py:161: UserWarning: lsoda: Repeated convergence failures (perhaps bad Jacobian or tolerances).
  solver._y, solver.t = integrator.run(
100%|██████████| 10/10 [00:24<00:00,  2.41s/it]
